In [1]:
# ---------------------------------------------------------------------------
# INSTALL com uv - wheel CUDA pre-compilado (NADA e compilado)
# ---------------------------------------------------------------------------
# O notebook usa a API in-process do llama-cpp-python: uvicorn/servidor HTTP
# NAO sao necessarios aqui (so seriam para o modo servidor OpenAI-compativel).
#
# 1) Instala o uv (instalador rapido em Rust) via pip - pacote pequeno.
# 2) numpy <2.3: evita o conflito com o numba 0.61.2 pre-instalado no Colab.
# 3) llama-cpp-python CUDA (cu125) vem do release oficial do autor, como
#    wheel pronto: sem CMAKE_ARGS, sem FORCE_CMAKE, sem build de ~30-40 min.
#    T4 (compute capability 7.5) e suportada por wheels cu12x.
#
# Se o Colab migrar para CUDA 13, troque o URL abaixo para o release -cu130
# equivalente (mesmo padrao: v0.3.35-cu130/llama_cpp_python-0.3.35-...whl).

!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu125
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 925.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


In [2]:
# Create the target directory
!mkdir -p /content/models

# Download the model weights directly using huggingface_hub
!pip install -q huggingface_hub
!python3 -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='bartowski/Qwen_Qwen3.5-0.8B-GGUF', filename='Qwen_Qwen3.5-0.8B-Q4_K_L.gguf', local_dir='/content/models')"


Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:   0% 0.00/641M [00:00<?, ?B/s]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:   4% 23.8M/641M [00:01<00:37, 16.7MB/s,  286kB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  16% 103M/641M [00:02<00:05, 99.7MB/s, 7.19MB/s  ] 
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  38% 246M/641M [00:02<00:01, 262MB/s, 19.3MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  44% 284M/641M [00:02<00:01, 231MB/s, 25.5MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  91% 584M/641M [00:04<00:00, 277MB/s, 46.3MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  97% 620M/641M [00:04<00:00, 235MB/s, 51.1MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: reconstructing file:  62% 399M/641M [00:04<00:01, 127MB/s, 26.4MB/s  ] 
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: reconstructing file:  74% 476M/641M [00:04<00:00, 178MB/s, 33.1MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: reconstructing file:  85% 547M/641M [00:04<00:00, 231MB/s, 39.7MB/s  ]
Qwen_Qw

In [3]:
# ---------------------------------------------------------------------------
# MODELO QWEN GGUF - contexto 32k, KV quantizado, prefill otimizado
# ---------------------------------------------------------------------------
import os
import shutil

from llama_cpp import Llama

MODEL_PATH = "/content/models/Qwen_Qwen3.5-0.8B-Q4_K_L.gguf"

_LOCAL_PATH = "/content/models_local/Qwen_Qwen3.5-0.8B-Q4_K_L.gguf"
if str(MODEL_PATH).startswith("/content/drive/"):
    os.makedirs(os.path.dirname(_LOCAL_PATH), exist_ok=True)
    if not os.path.exists(_LOCAL_PATH):
        print("Copiando GGUF do Drive para /content (1x)...")
        shutil.copyfile(MODEL_PATH, _LOCAL_PATH)
        print("Copia concluida.")
    MODEL_PATH = _LOCAL_PATH


def criar_llm():
  """Cria UMA instancia do modelo.

  n_batch/n_ubatch = 2048: o prompt eval (prefill) e o custo dominante;
  batches maiores aproveitam muito melhor a T4. Se der OOM, volte para 512.
  """
  return Llama(
      model_path=MODEL_PATH,
      n_ctx=32768,
      n_batch=2048,
      n_ubatch=2048,
      n_gpu_layers=-32,
      verbose=False,
      seed=0,
  )


llm = criar_llm()
print("Instancia do modelo carregada. n_ctx:", llm.n_ctx())


Instancia do modelo carregada. n_ctx: 32768


In [4]:
# ===========================================================================
# CLASSIFICACAO: INPUT CACHE + STRUCTURED OUTPUT (1 chamada por documento)
# ===========================================================================
# INPUT CACHE (equivalente local ao prompt caching das APIs):
#   PROMPT_SISTEMA (persona + rubricas das 3 tarefas) e o BLOCO CACHEAVEL:
#   bytes identicos em TODAS as chamadas. O llama.cpp mantem esse prefixo no
#   KV cache da sessao e nas chamadas seguintes NAO o reavalia (prompt eval
#   ~0). Com 1 chamada por documento, o documento tambem e avaliado 1x so.
#
# STRUCTURED OUTPUT:
#   Nenhuma instrucao textual de formato no prompt. As tres respostas saem
#   em UM objeto JSON com 3 campos, forcado por SCHEMA_PROCESSO (gramatica
#   GBNF na amostragem).
# ===========================================================================
import json
import time

from llama_cpp.llama_grammar import LlamaGrammar

# ---------------------------------------------------------------------------
# Schema - UM objeto JSON com as 3 respostas (formato 100% via gramatica)
# ---------------------------------------------------------------------------
SCHEMA_PROCESSO = {
    "type": "object",
    "properties": {
        "midias_digitais": {"type": "integer", "enum": [0, 1]},
        "impugnacao_digital": {"type": "integer", "enum": [0, 1]},
        "classificacao": {"type": "integer", "enum": [1, 2, 3]},
    },
    "required": ["midias_digitais", "impugnacao_digital", "classificacao"],
}

# ---------------------------------------------------------------------------
# BLOCO CACHEAVEL (mensagem system) - mesmo texto em TODA a execucao
# ---------------------------------------------------------------------------
PROMPT_SISTEMA = """Você é um especialista em provas digitais no processo judicial brasileiro e um analista criterioso de textos processuais.

Você recebe o texto integral de um processo judicial e deve responder as perguntas numeradas como TAREFA 1, TAREFA 2 e TAREFA 3, cada uma de forma independente.

REGRAS GERAIS OBRIGATÓRIAS:
1. Não faça inferências.
2. Considere exclusivamente as informações expressamente presentes no texto fornecido.
3. Responda cada tarefa de forma independente, aplicando somente as regras da tarefa correspondente.
4. O campo midias_digitais corresponde à TAREFA 1, o campo impugnacao_digital à TAREFA 2 e o campo classificacao à TAREFA 3.

TAREFA 1 - EXISTÊNCIA DE PROVA DIGITAL:
Determinar se existe PROVA DIGITAL relacionada aos fatos discutidos no processo.
Considere apenas evidências digitais utilizadas para demonstrar, confirmar, refutar ou contextualizar os fatos controvertidos.
Considere como prova digital:
- mensagens de WhatsApp, Telegram, SMS ou similares;
- e-mails;
- capturas de tela (prints);
- fotografias digitais;
- vídeos;
- áudios;
- gravações;
- publicações em redes sociais;
- registros de sistemas;
- logs;
- metadados;
- dados extraídos de celulares, computadores ou outros dispositivos;
- arquivos eletrônicos apresentados como evidência dos fatos;
- conteúdo armazenado em serviços digitais.
NÃO considere como prova digital:
- processo eletrônico;
- petições eletrônicas;
- movimentações processuais;
- documentos assinados digitalmente;
- certificados digitais;
- assinaturas eletrônicas;
- intimações eletrônicas;
- documentos meramente digitalizados;
- referências ao sistema do tribunal;
- e-SAJ, PJe, Projudi ou sistemas equivalentes;
- atos processuais eletrônicos em geral.
IMPORTANTE:
A mera existência de documentos eletrônicos nos autos NÃO significa existência de prova digital.
Classifique como 1 somente quando houver evidência digital relacionada aos fatos discutidos no processo.
Escala da TAREFA 1:
0 = Não há evidência de prova digital relacionada aos fatos.
1 = Há evidência de prova digital relacionada aos fatos.

TAREFA 2 - IMPUGNAÇÃO ESPECÍFICA DA PROVA DIGITAL:
Determinar se alguma das partes apresentou impugnação específica contra uma prova digital.
Considere apenas manifestações expressas que questionem a própria confiabilidade da prova digital.
Exemplos de impugnação específica:
- questionamento da autenticidade;
- questionamento da integridade;
- questionamento da origem;
- questionamento da autoria;
- alegação de adulteração;
- alegação de manipulação;
- alegação de montagem;
- alegação de edição;
- alegação de ausência ou ruptura da cadeia de custódia;
- questionamento dos métodos de coleta ou extração;
- questionamento de metadados, logs ou hashes;
- questionamento da confiabilidade técnica da evidência digital.
Considere petições, manifestações, recursos, quesitos, pareceres técnicos ou outros documentos.
NÃO considere como impugnação específica:
- mera discordância sobre os fatos;
- negativa dos fatos alegados;
- alegação genérica de insuficiência probatória;
- alegação de falta de convencimento do juiz;
- alegação de que a prova não comprova determinada narrativa;
- discussão jurídica sem questionamento da confiabilidade da prova digital;
- pedido genérico de produção de provas.
IMPORTANTE:
A crítica ao conteúdo da prova não é necessariamente impugnação da prova digital.
Classifique como 1 apenas quando houver questionamento expresso da confiabilidade, autenticidade, integridade, origem ou obtenção da evidência digital.
Escala da TAREFA 2:
0 = Não há impugnação específica de prova digital.
1 = Há impugnação específica de prova digital.

TAREFA 3 - ADERÊNCIA AOS PARÂMETROS DE CONFIABILIDADE:
Avaliar a aderência da prova digital aos parâmetros de confiabilidade.
Analise exclusivamente as informações presentes no texto e avalie, quando aplicável:
- cadeia de custódia;
- origem e identificação da evidência;
- coleta ou extração;
- datas e responsáveis;
- preservação e armazenamento;
- transferências e acessos;
- cópias e análises;
- hashes e integridade;
- imagens forenses;
- metadados e logs;
- autenticidade;
- método e ferramentas utilizadas;
- documentação;
- auditabilidade;
- contraditório;
- inconsistências entre documentos.
REGRAS OBRIGATÓRIAS DA TAREFA 3:
1. Não faça inferências.
2. Considere apenas informações expressamente presentes no texto.
3. A ausência de documentação NÃO significa automaticamente descumprimento.
4. Porém, a ausência de indícios negativos também NÃO significa aderência.
5. Classifique como 1 apenas quando existirem elementos positivos expressamente descritos nos autos que demonstrem aderência aos parâmetros.
6. Não utilize presunções de regularidade.
7. Quando as informações necessárias para avaliação forem insuficientes, limitadas ou inconclusivas, classifique como 3.
8. Diferencie "não consta dos autos" de "foi demonstrado que não foi realizado".
9. Para classificar como 2, deve existir evidência objetiva de fragilidade, descumprimento, inconsistência relevante ou questionamento fundamentado.
Escala da TAREFA 3:
1 = Potencialmente Segue. Existem evidências positivas e suficientes de aderência aos parâmetros relevantes para a prova digital analisada.
2 = Potencialmente Não Segue. Existe evidência objetiva de descumprimento, fragilidade relevante, inconsistência ou comprometimento da confiabilidade da prova digital.
3 = Indecisivo. As informações presentes são insuficientes, limitadas ou inconclusivas para avaliar aderência ou descumprimento."""

INSTRUCAO_FINAL = "Realize as TAREFAS 1, 2 e 3 sobre o processo acima."

MAX_RESPONSE_TOKENS = 192
RESERVA_CONTEXTO = 256
CHUNK_MAX_TOKENS = 12000
CHUNK_OVERLAP_TOKENS = 400


def _extrair_json(content):
    try:
        return json.loads(content.strip())
    except (ValueError, TypeError):
        pass
    ini = content.find("{")
    fim = content.rfind("}")
    if ini != -1 and fim > ini:
        return json.loads(content[ini:fim + 1])
    raise ValueError(f"Resposta fora do schema (nao-JSON): {content!r}")


class Analisador:
    """Wrapper de UMA instancia do modelo (1 por thread no batch paralelo)."""

    def __init__(self, modelo):
        self.llm = modelo
        self._overhead = None

    # ------------------------------ tokens ------------------------------
    def tokenizar(self, texto):
        return self.llm.tokenize(texto.encode("utf-8"), add_bos=False)

    def texto(self, tokens):
        return self.llm.detokenize(tokens).decode("utf-8", errors="replace")

    def _overhead_tokens(self):
        if self._overhead is None:
            self._overhead = (
                len(self.tokenizar(PROMPT_SISTEMA))
                + len(self.tokenizar("TEXTO INTEGRAL DO PROCESSO:"))
                + 128
            )
        return self._overhead

    def orcamento(self):
        """Tokens disponiveis para o corpo do documento por chamada."""
        return (
            self.llm.n_ctx()
            - MAX_RESPONSE_TOKENS
            - RESERVA_CONTEXTO
            - self._overhead_tokens()
        )

    # --------------------------- classificacao ---------------------------
    def _perguntar(self, documento):
        """1 chamada unica: system = PROMPT_SISTEMA (cacheavel); user = doc."""
        user_prompt = f"""TEXTO INTEGRAL DO PROCESSO:

{documento}

{INSTRUCAO_FINAL}"""
        messages = [
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": user_prompt},
        ]
        try:
            response = self.llm.create_chat_completion(
                messages=messages,
                response_format={"type": "json_object", "schema": SCHEMA_PROCESSO},
                max_tokens=MAX_RESPONSE_TOKENS,
                temperature=0.0,
                seed=0,
            )
        except TypeError:
            grammar = LlamaGrammar.from_json_schema(json.dumps(SCHEMA_PROCESSO))
            response = self.llm.create_chat_completion(
                messages=messages,
                grammar=grammar,
                max_tokens=MAX_RESPONSE_TOKENS,
                temperature=0.0,
                seed=0,
            )
        conteudo = response["choices"][0]["message"]["content"]
        dados = _extrair_json(conteudo)
        return {
            "midias_digitais": dados.get("midias_digitais"),
            "impugnacao_digital": dados.get("impugnacao_digital"),
            "classificacao": dados.get("classificacao"),
        }

    def _montar_chunks(self, tokens, chunk_max=CHUNK_MAX_TOKENS,
                       overlap=CHUNK_OVERLAP_TOKENS):
        total = len(tokens)
        if total <= chunk_max:
            return [tokens]
        partes, i = [], 0
        while i < total:
            j = min(i + chunk_max, total)
            partes.append(tokens[i:j])
            if j == total:
                break
            i = max(j - overlap, i + 1)
        return partes

    def _agregar(self, parciais):
        """OR p/ binarios; pior caso (2 > 1 > 3) p/ classificacao."""
        def _vals(campo):
            return [p[campo] for p in parciais if p.get(campo) is not None]

        mid = _vals("midias_digitais")
        imp = _vals("impugnacao_digital")
        cls = _vals("classificacao")
        return {
            "midias_digitais": 1 if 1 in mid else (0 if mid else None),
            "impugnacao_digital": 1 if 1 in imp else (0 if imp else None),
            "classificacao": 2 if 2 in cls else (1 if 1 in cls else (3 if cls else None)),
        }

    def classificar(self, documento):
        """Classifica 1 processo em 1 chamada (ou por chunks se gigante).

        Retorna (midias, impugnacao, classificacao, info).
        info: tokens, modo, chunks, tempo_llm_s, motivo.
        """
        t0 = time.perf_counter()
        info = {"modo": "direto", "chunks": 1, "tokens": 0,
                "tempo_llm_s": 0.0, "motivo": None}
        try:
            tokens_doc = self.tokenizar(documento)
        except Exception as e:
            info["motivo"] = f"tokenize falhou: {e}"
            return None, None, None, info

        info["tokens"] = len(tokens_doc)
        orcamento = self.orcamento()

        if info["tokens"] <= orcamento:
            parcial = self._perguntar(documento)
        else:
            chunk_max = min(CHUNK_MAX_TOKENS, orcamento)
            partes = self._montar_chunks(tokens_doc, chunk_max)
            info["modo"] = "chunks"
            info["chunks"] = len(partes)
            print(
                f"    (doc grande: {info['tokens']} tok -> "
                f"{len(partes)} chunks de ~{chunk_max} tok)"
            )
            parciais = []
            for k, parte in enumerate(partes, start=1):
                print(f"    chunk {k}/{len(partes)} ...")
                parciais.append(self._perguntar(self.texto(parte)))
            parcial = self._agregar(parciais)

        info["tempo_llm_s"] = time.perf_counter() - t0
        if all(parcial[c] is None for c in
               ("midias_digitais", "impugnacao_digital", "classificacao")):
            info["motivo"] = "resposta sem campos validos"
        return (
            parcial["midias_digitais"],
            parcial["impugnacao_digital"],
            parcial["classificacao"],
            info,
        )


# Instancia padrao (usada pelo diagnostico e pelo worker 1 do batch)
analisador = Analisador(llm)


In [ ]:
import os
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    def tqdm(seq, **kw):
        return seq

# ---------------------------------------------------------------------------
# CONFIG / PASTAS / SAIDA
# ---------------------------------------------------------------------------
pasta_origem = "/content/drive/MyDrive/ocr_export"
output_csv_path = "/content/drive/MyDrive/process_classification_results.csv"
batch_size = 10

LINHA = "=" * 64
TRACO = "-" * 64

METRICAS = {
    "inicio": time.perf_counter(),
    "novos": 0,
    "feitos": 0,
    "ok": 0,
    "parciais": 0,
    "nulos": 0,
    "tokens_prompt": 0,
    "leitura_s": 0.0,
    "llm_s": 0.0,
    "chunks": 0,
}


def _copiar_dataset(origem, destino):
    if not (str(origem).startswith("/content/drive/") and os.path.exists(origem)):
        return origem
    if os.path.exists(destino):
        return destino
    print("Copiando dataset do Drive para /content (1x; leituras ficam rapidas)...")
    arquivos = []
    for raiz, _, nomes in os.walk(origem):
        for nome in nomes:
            src = os.path.join(raiz, nome)
            arquivos.append((src, os.path.join(destino, os.path.relpath(src, origem))))
    for _, dst in arquivos:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
    with ThreadPoolExecutor(max_workers=16) as ex:
        futs = [ex.submit(shutil.copy2, src, dst) for src, dst in arquivos]
        for _ in tqdm(as_completed(futs), total=len(futs), leave=False, desc="copy"):
            pass
    print(f"Copiados {len(arquivos)} arquivos para {destino}")
    return destino


root_folder = _copiar_dataset(pasta_origem, "/content/ocr_export")


def _ler_texto(pasta):
    partes = []
    for arquivo in sorted(os.listdir(pasta)):
        if not arquivo.lower().endswith(".txt"):
            continue
        caminho = os.path.join(pasta, arquivo)
        try:
            with open(caminho, "r", encoding="utf-8") as f:
                partes.append(f.read())
        except Exception as e:
            print(f"     ! erro ao ler {arquivo}: {e}")
    return "\n".join(partes)


def _formatar(media, imp, cls):
    return " | ".join("-" if v is None else str(v) for v in (media, imp, cls))


def _estimativas():
    decorrido = time.perf_counter() - METRICAS["inicio"]
    feitos = METRICAS["feitos"]
    restantes = max(METRICAS["novos"] - feitos, 0)
    taxa = feitos / decorrido if decorrido > 0 else 0.0
    eta_min = (restantes / taxa) / 60.0 if taxa > 0 else float("nan")
    tok_s = (
        METRICAS["tokens_prompt"] / METRICAS["llm_s"]
        if METRICAS["llm_s"] > 0 else float("nan")
    )
    return {"feitos": feitos, "restantes": restantes, "doc_min": taxa * 60.0,
            "eta_min": eta_min, "tok_s": tok_s}


def _resumo_batch():
    e = _estimativas()
    print(TRACO)
    print(
        f"  OK {METRICAS['ok']} | PARCIAL {METRICAS['parciais']} | "
        f"NULO {METRICAS['nulos']} | chunks {METRICAS['chunks']}"
    )
    print(
        f"  VELOCIDADE: {e['doc_min']:.2f} doc/min | prefill ~{e['tok_s']:.0f} tok/s | "
        f"ETA ~{e['eta_min']:.0f} min"
    )
    print(
        f"  ACUMULADO: leitura {METRICAS['leitura_s']:.0f}s | llm {METRICAS['llm_s']:.0f}s"
    )
    print(TRACO)


if not os.path.exists(root_folder):
    print(f"ERROR: pasta nao encontrada:\n{root_folder}")
else:
    existing_df = pd.DataFrame()
    if os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path)
        print(f"Carregados {len(existing_df)} resultados existentes (resume).")

    if not existing_df.empty and "Process ID" in existing_df.columns:
        processados = set(existing_df["Process ID"].astype(str).tolist())
    else:
        processados = set()

    todas_pastas = [
        p for p in os.listdir(root_folder)
        if os.path.isdir(os.path.join(root_folder, p))
    ]
    novas = [p for p in todas_pastas if p not in processados]
    METRICAS["novos"] = len(novas)

    print(LINHA)
    print(
        f"  PROCESSOS: {len(todas_pastas)} total | {len(novas)} novos | "
        f"{len(todas_pastas) - len(novas)} ja feitos"
    )
    print(LINHA)

    if not novas:
        print("  Nada novo para processar. Encerrando.")

    total_batches = (len(novas) + batch_size - 1) // batch_size if novas else 0

    for i in range(0, len(novas), batch_size):
        batch_pastas = novas[i:i + batch_size]
        resultados_batch = []
        num = i // batch_size + 1
        t_batch = time.perf_counter()

        print("\n" + LINHA)
        print(f"  BATCH {num}/{total_batches}  ({len(batch_pastas)} processos)")
        print(LINHA)

        for folder in tqdm(batch_pastas, desc=f"batch {num}", leave=False):
            t_doc0 = time.perf_counter()
            caminho = os.path.join(root_folder, folder)

            texto = _ler_texto(caminho)
            dt_leitura = time.perf_counter() - t_doc0
            METRICAS["leitura_s"] += dt_leitura

            media = imp = cls = None
            n_tokens = 0
            n_chunks = 0
            llm_s = 0.0
            modo = "vazio"
            motivo = None
            if texto.strip():
                try:
                    media, imp, cls, info = analisador.classificar(texto)
                    n_tokens = info.get("tokens", 0)
                    n_chunks = info.get("chunks", 1)
                    llm_s = info.get("tempo_llm_s", 0.0)
                    modo = info.get("modo", "?")
                    motivo = info.get("motivo")
                except Exception as e:
                    modo = "erro"
                    motivo = str(e)
                    print(f"     ERRO inesperado ({folder}): {e}")

            METRICAS["feitos"] += 1
            METRICAS["llm_s"] += llm_s
            METRICAS["tokens_prompt"] += n_tokens
            METRICAS["chunks"] += n_chunks
            if media is None and imp is None and cls is None:
                METRICAS["nulos"] += 1
                simbolo = "X"
            elif None in (media, imp, cls):
                METRICAS["parciais"] += 1
                simbolo = "!"
            else:
                METRICAS["ok"] += 1
                simbolo = "ok"

            resultados_batch.append({
                "Process ID": folder,
                "MIDIAS_DIGITAIS": media,
                "IMPUGNACAO_DA_PROVA_DIGITAL": imp,
                "CLASSIFICACAO": cls,
            })

            dt_total = time.perf_counter() - t_doc0
            modo_txt = {
                "direto": "1x", "chunks": f"chunk x{n_chunks}",
                "vazio": "sem txt", "erro": "erro",
            }.get(modo, str(modo))
            if motivo:
                modo_txt += f" ({motivo})"
            print(
                f"  [{simbolo}] {folder[:40]:<40} "
                f"{_formatar(media, imp, cls):<11} "
                f"{dt_total:6.1f}s (llm {llm_s:4.1f}s, leitura {dt_leitura:4.2f}s)  "
                f"{n_tokens / 1000:6.1f}k tok  {modo_txt}"
            )

        if resultados_batch:
            df_batch = pd.DataFrame(resultados_batch)
            existe = os.path.exists(output_csv_path)
            df_batch.to_csv(
                output_csv_path,
                index=False,
                mode="a" if existe else "w",
                header=not existe,
            )
            print(f"\n  Batch salvo -> {output_csv_path}")

        print(f"  Tempo do batch {num}: {time.perf_counter() - t_batch:.1f}s")
        _resumo_batch()

    # -------------------------------------------------------------------
    # relatorio final
    # -------------------------------------------------------------------
    if os.path.exists(output_csv_path):
        final = pd.read_csv(output_csv_path)
        e = _estimativas()
        print("\n" + LINHA)
        print("  PROCESSAMENTO COMPLETO")
        print(
            f"  CSV: {output_csv_path} | linhas {len(final)} | "
            f"processos unicos {final['Process ID'].nunique()}"
        )
        print(LINHA)
        print("  Contagem CLASSIFICACAO (CSV acumulado):")
        print(final["CLASSIFICACAO"].value_counts(dropna=False).to_string())
        print(LINHA)
        tabela = pd.DataFrame([{
            "novos": e["feitos"],
            "ok": METRICAS["ok"],
            "parcial": METRICAS["parciais"],
            "nulo": METRICAS["nulos"],
            "tokens_prefill": METRICAS["tokens_prompt"],
            "tempo_llm_s": round(METRICAS["llm_s"], 1),
            "tempo_leitura_s": round(METRICAS["leitura_s"], 1),
            "doc/min": round(e["doc_min"], 2),
            "prefill tok/s": (None if e["tok_s"] != e["tok_s"] else round(e["tok_s"], 0)),
        }])
        print("  ESTATISTICAS DA EXECUCAO:")
        print(tabela.to_string(index=False))
        print(LINHA)
    else:
        print("  Nenhum CSV gerado.")


Copiando dataset do Drive para /content (1x; leituras ficam rapidas)...


copy:   0%|          | 0/2867 [00:00<?, ?it/s]

Copiados 2867 arquivos para /content/ocr_export
Carregados 50 resultados existentes (resume).
  PROCESSOS: 349 total | 299 novos | 50 ja feitos

  BATCH 1/30  (10 processos)


batch 1:   0%|          | 0/10 [00:00<?, ?it/s]

  [ok] process_1011465-59.2018.8.26.0292        0 | 0 | 1      2.9s (llm  2.9s, leitura 0.00s)     4.6k tok  1x
  [ok] process_1006032-53.2019.8.26.0223        1 | 1 | 1      6.0s (llm  6.0s, leitura 0.00s)    13.1k tok  1x
  [ok] process_1001062-33.2019.8.26.0280        0 | 0 | 1      3.7s (llm  3.7s, leitura 0.00s)     4.5k tok  1x
  [ok] process_1001431-07.2019.8.26.0222        0 | 0 | 3      3.3s (llm  3.3s, leitura 0.00s)     5.6k tok  1x
  [ok] process_1007722-41.2018.8.26.0292        1 | 0 | 1      5.7s (llm  5.7s, leitura 0.00s)    11.9k tok  1x
  [ok] process_1000984-61.2017.8.26.0651        0 | 0 | 1      6.0s (llm  6.0s, leitura 0.00s)    10.6k tok  1x
  [ok] process_1006750-71.2017.8.26.0077        1 | 0 | 1      4.6s (llm  4.6s, leitura 0.00s)     8.7k tok  1x
  [ok] process_1007336-45.2017.8.26.0292        1 | 0 | 1      3.7s (llm  3.7s, leitura 0.00s)     6.7k tok  1x
  [ok] process_1001718-75.2018.8.26.0651        1 | 0 | 1      6.1s (llm  6.1s, leitura 0.00s)    10.6k 

...

  [ok] process_1010569-79.2019.8.26.0292        0 | 0 | 3      3.2s (llm  3.2s, leitura 0.00s)     5.9k tok  1x
  [ok] process_1009927-09.2019.8.26.0292        0 | 0 | 1      5.4s (llm  5.4s, leitura 0.00s)    10.0k tok  1x
  [ok] process_1003060-68.2017.8.26.0292        1 | 1 | 1      4.4s (llm  4.4s, leitura 0.00s)     8.7k tok  1x
  [ok] process_1005259-08.2019.8.26.0223        1 | 1 | 1      9.0s (llm  9.0s, leitura 0.00s)    17.1k tok  1x
  [ok] process_1009718-40.2019.8.26.0292        1 | 0 | 1      5.0s (llm  5.0s, leitura 0.00s)    10.2k tok  1x
  [ok] process_1001941-09.2016.8.26.0292        0 | 0 | 1      2.8s (llm  2.8s, leitura 0.00s)     4.2k tok  1x
  [ok] process_1001054-56.2019.8.26.0280        1 | 1 | 1      4.8s (llm  4.8s, leitura 0.00s)     9.6k tok  1x
  [ok] process_1002545-52.2019.8.26.0651        0 | 0 | 3      4.8s (llm  4.8s, leitura 0.00s)     8.6k tok  1x
  [ok] process_1005346-87.2014.8.26.0077        0 | 0 | 1      3.1s (llm  3.1s, leitura 0.00s)     5.3k 

batch 6:   0%|          | 0/10 [00:00<?, ?it/s]

  [ok] process_1003386-95.2019.8.26.0441        0 | 0 | 1      6.6s (llm  6.6s, leitura 0.00s)    12.4k tok  1x
  [ok] process_1009637-28.2018.8.26.0292        0 | 0 | 1      3.6s (llm  3.6s, leitura 0.00s)     5.8k tok  1x
  [ok] process_1002132-44.2016.8.26.0651        0 | 0 | 1      7.4s (llm  7.4s, leitura 0.00s)    13.7k tok  1x
  [ok] process_1008674-54.2017.8.26.0292        0 | 0 | 1      3.9s (llm  3.9s, leitura 0.00s)     6.4k tok  1x
  [ok] process_1006525-51.2018.8.26.0292        1 | 0 | 1      4.5s (llm  4.5s, leitura 0.00s)     8.6k tok  1x
  [ok] process_1001510-97.2023.8.26.0269        0 | 0 | 1      2.9s (llm  2.9s, leitura 0.00s)     3.6k tok  1x
  [ok] process_1004784-39.2017.8.26.0541        0 | 0 | 1      4.5s (llm  4.5s, leitura 0.00s)     7.6k tok  1x
  [ok] process_1007821-41.2022.8.26.0269        0 | 0 | 3      2.2s (llm  2.2s, leitura 0.00s)     2.1k tok  1x
  [ok] process_1003491-68.2018.8.26.0292        0 | 0 | 1      3.8s (llm  3.8s, leitura 0.00s)     7.4k 

batch 7:   0%|          | 0/10 [00:00<?, ?it/s]

  [ok] process_1011666-81.2022.8.26.0269        0 | 0 | 1      5.2s (llm  5.2s, leitura 0.00s)    10.6k tok  1x
  [ok] process_1010589-75.2016.8.26.0292        0 | 0 | 1      4.2s (llm  4.1s, leitura 0.00s)     8.5k tok  1x
  [ok] process_1002588-68.2019.8.26.0366        0 | 0 | 1      5.1s (llm  5.1s, leitura 0.00s)    10.4k tok  1x
  [ok] process_1000984-95.2016.8.26.0651        0 | 0 | 1      5.2s (llm  5.2s, leitura 0.00s)    10.2k tok  1x
